# Нагрузочное тестирование сервиса

Сервис должен быть поднят (`uvicorn app1:app --port 8079` или docker compose).
Параметры прогона задаются переменными окружения: `LOAD_TEST_URL`, `LOAD_TEST_WORKERS`,
`LOAD_TEST_MAX_TIME` (секунды), `LOAD_TEST_SAMPLES`, `LOAD_TEST_P95_SLO_MS`,
`LOAD_TEST_MIN_SUCCESS_RATE`. По итогам формируются `load_test_report.html` и
`load_test_report.png`; при нарушении SLO ячейка падает с AssertionError.

In [ ]:
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from IPython.display import HTML

# --- Параметры прогона (переопределяются переменными окружения) ---
URL = os.getenv('LOAD_TEST_URL', 'http://localhost:8079/predict')
WORKERS = int(os.getenv('LOAD_TEST_WORKERS', '50'))
MAX_TIME = float(os.getenv('LOAD_TEST_MAX_TIME', '50'))
N_PROFILES = int(os.getenv('LOAD_TEST_SAMPLES', '100'))
# SLO: тест падает, если p95 выше или доля успешных ответов ниже порога
P95_SLO_MS = float(os.getenv('LOAD_TEST_P95_SLO_MS', '5000'))
MIN_SUCCESS_RATE = float(os.getenv('LOAD_TEST_MIN_SUCCESS_RATE', '99.0'))

print(f'URL: {URL}, workers: {WORKERS}, max_time: {MAX_TIME}с, профилей: {N_PROFILES}')
print(f'SLO: p95 < {P95_SLO_MS} мс, success_rate >= {MIN_SUCCESS_RATE}%')


class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)


def load_profiles(path='data/train_ver2.csv', n=N_PROFILES, seed=42):
    """Реальные профили клиентов — целые строки датасета.

    Раньше профиль собирался из несовместимых значений разных строк
    (возраст 116 + сегмент «UNIVERSITARIO») — такие запросы нерепрезентативны
    (CODE_REVIEW §5.3). Читаем только первые 100k строк: для выборки достаточно.
    """
    sample = pd.read_csv(path, nrows=100_000).sample(n, random_state=seed)
    return sample.to_dict('records')


def check_response_schema(payload):
    """Проверяет схему ответа POST /predict; возвращает текст ошибки или None."""
    if not isinstance(payload, dict):
        return f'ответ не dict: {type(payload)}'
    for key in ('prediction', 'product', 'confidence', 'top_k'):
        if key not in payload:
            return f'нет ключа {key!r} в ответе: {payload}'
    if not isinstance(payload['prediction'], int):
        return f"prediction не int: {payload['prediction']!r}"
    if payload['prediction'] not in label_map:
        return f"неизвестный класс {payload['prediction']}"
    if not isinstance(payload['top_k'], list) or not payload['top_k']:
        return 'top_k пуст или не список'
    if payload['top_k'][0]['code'] != payload['prediction']:
        return 'top_k[0] не совпадает с prediction'
    probabilities = [item['probability'] for item in payload['top_k']]
    if probabilities != sorted(probabilities, reverse=True):
        return 'top_k не отсортирован по убыванию вероятности'
    return None


def send_request(url, profile, timeout=30):
    """Один запрос к сервису: возвращает (успех, латентность, ошибку)."""
    started = time.time()
    try:
        response = requests.post(
            url,
            headers={'Content-Type': 'application/json'},
            data=json.dumps(profile, cls=NumpyEncoder),
            timeout=timeout,
        )
        latency = (time.time() - started) * 1000  # мс
        if response.status_code != 200:
            return {'status': response.status_code, 'latency': latency,
                    'success': False, 'error': f'HTTP {response.status_code}'}
        schema_error = check_response_schema(response.json())
        return {'status': 200, 'latency': latency,
                'success': schema_error is None, 'error': schema_error}
    except requests.RequestException as exc:
        return {'status': 0, 'latency': (time.time() - started) * 1000,
                'success': False, 'error': f'{type(exc).__name__}: {exc}'}


def run_load_test(profiles, url, max_time, workers):
    """Шлёт запросы параллельно через пул воркеров, пока не выйдет время.

    Раньше Executor создавался, но запросы шли последовательно в цикле `while`
    (~1.8 RPS), и p95/p99 ничего не значили (CODE_REVIEW §5.1). Теперь фьючерсы
    реально выполняются конкурентно, а очередь ограничена числом воркеров.
    """
    results, profile_index, submitted = [], 0, set()
    started = time.time()
    with ThreadPoolExecutor(max_workers=workers) as executor:
        while (time.time() - started) < max_time or submitted:
            while len(submitted) < workers and (time.time() - started) < max_time:
                future = executor.submit(send_request, url, profiles[profile_index % len(profiles)])
                submitted.add(future)
                profile_index += 1
            done, submitted = set(), submitted
            for future in as_completed(submitted, timeout=max_time):
                done.add(future)
                try:
                    results.append(future.result())
                except Exception as exc:  # падение воркера — тоже результат
                    results.append({'status': 0, 'latency': 0.0, 'success': False,
                                    'error': f'{type(exc).__name__}: {exc}'})
                submitted = submitted - done
                break  # добрать очередь до workers на следующей итерации
            if profile_index % 50 == 0:
                print(f'  отправлено {profile_index}, готово {len(results)}')
    frame = pd.DataFrame(results)
    frame.attrs['wall_seconds'] = time.time() - started
    return frame


def generate_report(results_df):
    """Считает метрики, рисует графики, пишет HTML-отчёт. Падает при нарушении SLO."""
    ok = results_df['success'].mean() * 100
    latencies = results_df.loc[results_df['success'], 'latency']
    if latencies.empty:  # все запросы упали — латентности нет, упадём по success_rate ниже
        latencies = pd.Series([0.0])
    wall_seconds = results_df.attrs.get('wall_seconds', 0)
    report = {
        'total_requests': len(results_df),
        'success_rate': ok,
        'avg_latency': float(latencies.mean()),
        'p95_latency': float(np.percentile(latencies, 95)),
        'p99_latency': float(np.percentile(latencies, 99)),
        'rps': len(results_df) / wall_seconds if wall_seconds else None,
        'error_distribution': results_df['error'].value_counts().to_dict(),
    }

    plt.figure(figsize=(15, 5))
    plt.subplot(131)
    sns.histplot(results_df['latency'], bins=50)
    plt.title('Latency Distribution')
    plt.subplot(132)
    results_df['status'].value_counts().plot(kind='bar')
    plt.title('Status Code Distribution')
    plt.subplot(133)
    pd.Series([ok, 100 - ok], index=['Success', 'Error']).plot(
        kind='pie', autopct='%1.1f%%')
    plt.title('Success Rate')
    plt.tight_layout()
    plt.savefig('load_test_report.png')

    html_report = f"""
    <html>
        <body>
            <h1>Load Test Report</h1>
            <p>URL: {URL}, workers: {WORKERS}, SLO: p95 &lt; {P95_SLO_MS} мс,
               success_rate &gt;= {MIN_SUCCESS_RATE}%</p>
            <pre>{pd.DataFrame([report]).to_html()}</pre>
            <h2>Charts</h2>
            <img src="load_test_report.png" width="100%">
        </body>
    </html>
    """
    with open('load_test_report.html', 'w', encoding='utf-8') as file:
        file.write(html_report)

    assert ok >= MIN_SUCCESS_RATE, (
        f'SLO нарушен: success_rate={ok:.1f}% < {MIN_SUCCESS_RATE}%')
    assert report['p95_latency'] < P95_SLO_MS, (
        f"SLO нарушен: p95={report['p95_latency']:.0f} мс >= {P95_SLO_MS} мс")
    return report


# --- Подготовка ---
try:
    with open('fastapi/preprocessing_params.json', encoding='utf-8') as file:
        label_map = {int(code) for code in json.load(file)['label_map']}
except (OSError, KeyError):
    label_map = {0, 99}  # минимум: классы «нет покупки» и «редкий продукт»
    print('preprocessing_params.json не найден — проверяю только базовые классы')

profiles = load_profiles()
print(f'Загружено профилей: {len(profiles)}')

# --- Прогон ---
print('Starting load test...')
results = run_load_test(profiles, URL, max_time=MAX_TIME, workers=WORKERS)

print('Test completed, generating report...')
report = generate_report(results)

display(HTML(pd.DataFrame([report]).to_html()))
display(HTML('<h2>First 5 responses:</h2>'))
display(results.head().to_html())


In [ ]:
# Повторный прогон с другими параметрами — без дублирования кода (CODE_REVIEW §5.2).
# Для долгого прогона: LOAD_TEST_MAX_TIME=505, LOAD_TEST_WORKERS=50.
results = run_load_test(profiles, URL, max_time=float(os.getenv('LOAD_TEST_MAX_TIME', '50')),
                        workers=int(os.getenv('LOAD_TEST_WORKERS', '50')))
report = generate_report(results)
display(HTML(pd.DataFrame([report]).to_html()))
